**Exercise 9 - Object Detection [10 pt]** 

---
**NOTE:** If you have not yet completed tasks 9.1 - 9.4, first open *ex9_object_detection_dev.ipynb* and complete them there.

This notebook deals with the training of the model you developed in *ex9_object_detection_dev.ipynb*. To perform the training on Google Colab, do the following steps:
1. Upload the directory *mlrcv/*, as well as this notebook *ex9_object_detection_training.ipynb* to Google Drive. Create a directory *ex9_data/ex9/* on your Google Drive main directory and upload everything there.
2. In Google Drive: Right click on this notebook *ex9_object_detection_training.ipynb* and open with Google Colab. Perform the following steps in this notebook within Google Colab.
3. Change your runtime type (upper left corner): *Hardware accelerator* should be a GPU and *runtime version* should be latest.

Now, run the follwing cell and follow the described steps to allow the Google Colab to access this data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir center_logs
!mkdir checkpoint
!mkdir data
!mkdir mlrcv

!cp -r /content/drive/MyDrive/ex9_data/ex9/mlrcv/* ./mlrcv/

**Import dependencies**

---
Run the following cell to import all dependencies:

In [ ]:
!pip3 install pytorch_lightning
import torch
import torchvision
from pytorch_lightning import Trainer
from mlrcv.model import CenterNet
from mlrcv.pre_process import *
from mlrcv.collation import *
from mlrcv.loss import *
from mlrcv.trainer import CenterNetTrainer
from mlrcv.utils import *
import matplotlib.pyplot as plt

**9.5 Training the network [2 pt]**

---

Now that you have implemented the necessary modules you run the following code to train the network. The training steps are logged using tensorboard, which can be started inside notebooks, so you can keep track of the training/validation losses and image bounding box predictions.

In [ ]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05, hue=0.05)
])

train_data = torchvision.datasets.VOCDetection('./data', year='2007', image_set='train', transform=transform, download=True)
train_loader = torch.utils.data.DataLoader(train_data,
                                        batch_size=16,
                                        shuffle=True,
                                        collate_fn=VOCollation(),
                                        num_workers=2)

test_data = torchvision.datasets.VOCDetection('./data', year='2007', image_set='test', download=True)
test_loader = torch.utils.data.DataLoader(test_data,
                                        batch_size=16,
                                        collate_fn=VOCollation(),
                                        shuffle=False,
                                        num_workers=2)

Before start the training, start tensorboard. The logs will be saved in the *center_logs/*, so you can start tensorboard inside the notebook by executing the following cell:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir center_logs

Now you can start the training and follow the training through tensorboard.

In [ ]:
# lr same as paper
# more epochs because we have a smaller dataset, so we need more iterations
lr = 5e-4
epochs = 500

model = CenterNet().cuda()
centernet = CenterNetTrainer(model, centerloss, (1.0, 0.5), train_loader, test_loader, lr, epochs)

trainer = Trainer(max_epochs=epochs, check_val_every_n_epoch=10)
trainer.fit(centernet)

Note that the training may take around 2h~3h to bring relevant results, that's why it's important to check if every previous task have been correctly implemented and to keep track of the logs, so you can stop the training once you notice already that something is not right. 

After around 150 epochs you should see on tensorboard the mIoU metric increasing from close to 0% to around 60%. 

If you run out of compute on Google Colab, don't worry. The final goal of this task is not to achieve any specific accuracy, but to get the training pipeline running. Just download the generated *predictions.npz* from the Colab and place it inside your local *ex09/*-directory. This file is generated every 10th epoch. We will not check for any specific accuracy, but rather if the file was generated correctly.

The directory structure for your final submission should then look like this:
```
ex09/
├── ex09_object_detection_dev.ipynb
├── ex09_object_detection_training.ipynb
├── predictions.npz
├── mlrcv/
│   ├── collation.py
│   ├── ...
```

**Assignment Submission**

---

To submit your assignment, run the following command from the root of the repository (For Linux and MacOS):

```bash
sh submit.sh
```

For Windows, please use `submit.bat` or try git bash.

**Check That You Actually Submitted**

You can check that your submission worked in several ways:

1. There is a new line in the file submission.txt with current date and time.
2. There is a new commit in your git history with the title "Submission".